# FireST-GraphNet Multi-Task Training on Colab

This notebook trains the FireST-GraphNet model on three tasks:
- **AF**: Active Fire Detection
- **BA**: Burned Area Mapping  
- **Pred**: Fire Progression Prediction

---

## Prerequisites

Before running this notebook:
1. ✅ Regenerate datasets using correct parameters (see DATA_REGENERATION_STEP_BY_STEP.md)
2. ✅ Upload datasets to Google Drive under `FireST-GraphNet/`
3. ✅ Verify data with `quick_verify.py` locally

---

## Step 1: Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install additional dependencies if needed
!pip install torch-geometric
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.0.0+cu118.html

## Step 2: Mount Google Drive and Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import shutil
from pathlib import Path

# Path to data on Google Drive
DRIVE_DATA_PATH = Path('/content/drive/MyDrive/FireST-GraphNet/FireST-GraphNet-Clean')

# Check if data exists
if not DRIVE_DATA_PATH.exists():
    raise FileNotFoundError(f"Data not found at {DRIVE_DATA_PATH}. Please upload datasets to Google Drive first.")

print("✓ Google Drive data found")
print(f"  Path: {DRIVE_DATA_PATH}")

# List available datasets
for folder in ['dataset_train', 'dataset_val', 'dataset_test']:
    folder_path = DRIVE_DATA_PATH / folder
    if folder_path.exists():
        files = list(folder_path.glob('*.npy'))
        total_size = sum(f.stat().st_size for f in files) / (1024**3)  # GB
        print(f"  ✓ {folder}: {len(files)} files ({total_size:.2f} GB)")
    else:
        print(f"  ✗ {folder}: not found")

### Option A: Copy Data to Colab Local Storage (Faster I/O)

**Recommended** if you have enough Colab storage (~20-30 GB).

In [ ]:
# Copy data to local /content for faster access
LOCAL_DATA_PATH = Path('/content/FireST-GraphNet')
LOCAL_DATA_PATH.mkdir(exist_ok=True)

print("Copying datasets from Drive to local storage...")
print("This will take 5-10 minutes but speeds up training significantly.\n")

for dataset in ['dataset_train', 'dataset_val']:
    src = DRIVE_DATA_PATH / dataset
    dst = LOCAL_DATA_PATH / dataset
    
    if src.exists():
        if dst.exists():
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"✓ Copied {dataset}")

print("\n✓ Data copy complete!")

# Use local path for training
DATA_ROOT = LOCAL_DATA_PATH

### Option B: Use Data Directly from Google Drive (Slower but Saves Space)

In [ ]:
# Use data directly from Drive (slower I/O)
# Uncomment this if you want to use Drive directly instead of copying

# DATA_ROOT = DRIVE_DATA_PATH
# print(f"Using data directly from Google Drive: {DATA_ROOT}")

## Step 3: Clone FireST-GraphNet Code

In [ ]:
# Clone your repository (replace with your repo URL)
!git clone https://github.com/YOUR_USERNAME/FireST-GraphNet.git /content/code

# Or upload code from Drive if you saved it there
# CODE_PATH = DRIVE_DATA_PATH / 'code'
# shutil.copytree(CODE_PATH, '/content/code')

%cd /content/code

## Step 4: Quick Data Verification

In [ ]:
# Run quick verification
import sys
sys.path.insert(0, '/content/code')

import numpy as np

train_dir = DATA_ROOT / 'dataset_train'

# Load files
img_files = list(train_dir.glob('*_img_*.npy'))
label_files = list(train_dir.glob('*_label_*.npy'))
firepred_files = list(train_dir.glob('firepred_*.npy'))
lulc_files = list(train_dir.glob('lulc_*.npy'))

print("Dataset files:")
print(f"  img:      {img_files[0].name if img_files else 'NOT FOUND'}")
print(f"  label:    {label_files[0].name if label_files else 'NOT FOUND'}")
print(f"  firepred: {firepred_files[0].name if firepred_files else 'NOT FOUND'}")
print(f"  lulc:     {lulc_files[0].name if lulc_files else 'NOT FOUND'}")

if all([img_files, label_files, firepred_files, lulc_files]):
    # Check shapes and N consistency
    img = np.load(str(img_files[0]), mmap_mode='r')
    label = np.load(str(label_files[0]), mmap_mode='r')
    firepred = np.load(str(firepred_files[0]), mmap_mode='r')
    lulc = np.load(str(lulc_files[0]), mmap_mode='r')
    
    print(f"\nData shapes:")
    print(f"  img:      {img.shape}")
    print(f"  label:    {label.shape}")
    print(f"  firepred: {firepred.shape}")
    print(f"  lulc:     {lulc.shape}")
    
    N_match = (img.shape[0] == firepred.shape[0] == lulc.shape[0])
    print(f"\nN consistency: {'✓ PASS' if N_match else '✗ FAIL (CRITICAL)'}")
    
    if N_match:
        print("\n✓ Data looks good! Ready for training.")
    else:
        print("\n✗ CRITICAL: N mismatch detected! Data needs regeneration.")
        print("   See DATA_REGENERATION_STEP_BY_STEP.md")
else:
    print("\n✗ Missing dataset files!")

## Step 5: Configure Training

Choose which task to train:

In [ ]:
# Choose task: 'af', 'ba', or 'pred'
TASK = 'af'  # Active Fire Detection (currently the only fully supported task)

# Training config
CONFIG = {
    'task': {
        'name': TASK,
    },
    'data': {
        'root': str(DATA_ROOT / 'dataset_train'),
        'T_in': 9,
        'T_out': 1,
        'label_channel': 1,  # AF uses channel 1
        'num_workers': 2,
    },
    'model': {
        'in_channels': 8,
        'out_channels': 1,
        'encoder_channels': [32, 64, 128],
        'decoder_channels': [128, 64, 32],
    },
    'training': {
        'batch_size': 8,
        'epochs': 30,
        'lr': 0.0003,
        'weight_decay': 0.001,
        'pos_weight': 10.0,
        'mixed_precision': True,
    },
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'output_dir': f'/content/outputs/{TASK}_run',
}

print(f"Training configuration for task: {TASK}")
print(f"  Data root: {CONFIG['data']['root']}")
print(f"  Device: {CONFIG['device']}")
print(f"  Epochs: {CONFIG['training']['epochs']}")
print(f"  Batch size: {CONFIG['training']['batch_size']}")

## Step 6: Train the Model

In [ ]:
# Import training modules
from firest_graphnet.data.dataset import TSSatFireSequenceDataset, build_datasets
from firest_graphnet.models.firest_graphnet import FireSTGraphNet
from firest_graphnet.training.trainer import Trainer
from torch.utils.data import DataLoader

# Build datasets
print("Building datasets...")
train_ds, val_ds = build_datasets(
    data_root=str(DATA_ROOT),
    T_in=CONFIG['data']['T_in'],
    T_out=CONFIG['data']['T_out'],
    label_channel=CONFIG['data']['label_channel'],
)

print(f"  Train samples: {len(train_ds)}")
print(f"  Val samples: {len(val_ds)}")

# Create dataloaders
train_loader = DataLoader(
    train_ds,
    batch_size=CONFIG['training']['batch_size'],
    shuffle=True,
    num_workers=CONFIG['data']['num_workers'],
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=CONFIG['training']['batch_size'],
    shuffle=False,
    num_workers=CONFIG['data']['num_workers'],
    pin_memory=True,
)

print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")

In [ ]:
# Initialize model
print("\nInitializing model...")
model = FireSTGraphNet(**CONFIG['model'])

device = torch.device(CONFIG['device'])
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / 1024**2:.1f} MB")

In [ ]:
# Create trainer
print("\nCreating trainer...")
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=CONFIG,
    device=device,
    output_dir=CONFIG['output_dir'],
)

print("✓ Trainer ready")
print(f"  Output directory: {CONFIG['output_dir']}")

In [ ]:
# Start training!
print("\n" + "="*70)
print(f"STARTING TRAINING: {TASK.upper()} Task")
print("="*70 + "\n")

trainer.fit()

## Step 7: Evaluate Results

In [ ]:
# Load training metrics
import pandas as pd
import matplotlib.pyplot as plt

metrics_csv = Path(CONFIG['output_dir']) / 'metrics.csv'
if metrics_csv.exists():
    df = pd.read_csv(metrics_csv)
    
    # Plot training curves
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # Loss
    axes[0, 0].plot(df['epoch'], df['train_loss'], label='Train')
    axes[0, 0].plot(df['epoch'], df['val_loss'], label='Val')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss Curve')
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # IoU
    axes[0, 1].plot(df['epoch'], df['train_iou'], label='Train')
    axes[0, 1].plot(df['epoch'], df['val_iou'], label='Val')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('IoU')
    axes[0, 1].set_title('IoU Curve')
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # F1
    axes[1, 0].plot(df['epoch'], df['train_f1'], label='Train')
    axes[1, 0].plot(df['epoch'], df['val_f1'], label='Val')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1')
    axes[1, 0].set_title('F1 Score Curve')
    axes[1, 0].legend()
    axes[1, 0].grid(True)
    
    # Learning rate
    axes[1, 1].plot(df['epoch'], df['lr'])
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(Path(CONFIG['output_dir']) / 'training_curves.png', dpi=150)
    plt.show()
    
    # Print best metrics
    best_epoch = df['val_iou'].idxmax()
    print(f"\n{'='*70}")
    print("BEST MODEL PERFORMANCE")
    print(f"{'='*70}")
    print(f"  Epoch: {df.loc[best_epoch, 'epoch']}")
    print(f"  Val Loss: {df.loc[best_epoch, 'val_loss']:.4f}")
    print(f"  Val IoU: {df.loc[best_epoch, 'val_iou']:.4f}")
    print(f"  Val F1: {df.loc[best_epoch, 'val_f1']:.4f}")
    print(f"  Val Precision: {df.loc[best_epoch, 'val_precision']:.4f}")
    print(f"  Val Recall: {df.loc[best_epoch, 'val_recall']:.4f}")
    print(f"{'='*70}\n")
else:
    print("Metrics file not found. Training may not have started yet.")

## Step 8: Save Results Back to Google Drive

In [ ]:
# Copy output files back to Google Drive
output_dir = Path(CONFIG['output_dir'])
drive_output_dir = DRIVE_DATA_PATH / 'outputs' / f"{TASK}_colab_run"
drive_output_dir.mkdir(parents=True, exist_ok=True)

print("Copying results to Google Drive...")

# Copy model checkpoint
if (output_dir / 'best_model.pt').exists():
    shutil.copy2(output_dir / 'best_model.pt', drive_output_dir / 'best_model.pt')
    print("  ✓ Copied best_model.pt")

# Copy metrics
if (output_dir / 'metrics.csv').exists():
    shutil.copy2(output_dir / 'metrics.csv', drive_output_dir / 'metrics.csv')
    print("  ✓ Copied metrics.csv")

# Copy training curves
if (output_dir / 'training_curves.png').exists():
    shutil.copy2(output_dir / 'training_curves.png', drive_output_dir / 'training_curves.png')
    print("  ✓ Copied training_curves.png")

print(f"\n✓ Results saved to: {drive_output_dir}")

---

## Training Complete!

- Model checkpoint: `best_model.pt`
- Training logs: `metrics.csv`
- Visualizations: `training_curves.png`

All results have been saved to your Google Drive for future use.